# Random Forest

In this notebook, we will be implementing and tuning a Random Forest model to see how well it performs on our training set. 

In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler

from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone

from sklearn.model_selection import StratifiedKFold, cross_val_score, RandomizedSearchCV

from sklearn.metrics import make_scorer, classification_report, confusion_matrix, accuracy_score, roc_auc_score, roc_curve
from scipy.stats import randint, uniform

import matplotlib.pyplot as plt
import seaborn as sns


In [3]:
path = ['../Data/X_train.csv', '../Data/y_train.csv']
X_train_temp, y_train_temp = [pd.read_csv(f, index_col=0) for f in path]


In [4]:
top_1000_genes_temp = pd.read_csv('../Data/top_1000_genes.csv', index_col=0)
top_1000_genes = list(top_1000_genes_temp['0'].copy())

In [5]:
X_train = X_train_temp[top_1000_genes].copy()
y_train= np.array(y_train_temp['Cluster'].copy())

In [6]:
X_train.shape

(610, 1000)

## Implement initial Random Forest

Model run on dataset with 1000 features to see how well it performs in general first. 

In [7]:
# Define the XGBoost classifier
rf_pipe = Pipeline([
    ('scale', StandardScaler()),
    ('model', RandomForestClassifier(
        n_estimators=100,
        max_depth=5,          # limit tree depth
        max_features="sqrt",  # random feature selection helps with p >> n
        random_state=42
    ))
])


# Define stratified k-fold cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Evaluate accuracy
acc_scores = cross_val_score(
    rf_pipe,
    X_train, y_train,
    cv=cv,
    scoring=make_scorer(accuracy_score)
)

print("Cross-validation accuracies:", acc_scores)
print("Mean accuracy: {:.3f} ± {:.3f}".format(acc_scores.mean(), acc_scores.std()))

roc_auc_scores = cross_val_score(
    rf_pipe,
    X_train, y_train,
    cv=cv,
    scoring='roc_auc'
)

print("Cross-validation ROC-AUCs:", roc_auc_scores)
print("Mean ROC-AUC: {:.3f} ± {:.3f}".format(roc_auc_scores.mean(), roc_auc_scores.std()))




Cross-validation accuracies: [0.89344262 0.89344262 0.90163934 0.90983607 0.94262295]
Mean accuracy: 0.908 ± 0.018
Cross-validation ROC-AUCs: [0.94285714 0.94449918 0.96156331 0.95639535 0.98514212]
Mean ROC-AUC: 0.958 ± 0.015


## Random Forest  with Nested CV
Hyperparameter tuning to find the best model

In [9]:
# Hyperparameter distributions for RandomizedSearchCV

# Parameter distributions (reasonable search space)
param_distribution = {
    'n_estimators': randint(50, 500),
    'max_depth': randint(2, 50),
    'max_features': uniform(0.1, 0.9),
    'min_samples_split': randint(2, 10), # minimum number of samples required to split an internal node
    'min_samples_leaf': randint(2, 10), # minimum number of samples required to be at a leaf node
    'bootstrap': [True, False]
}

# Using nested CV with inner and outer scores
outer_cv = StratifiedKFold(n_splits = 5, shuffle=True, random_state=123)
nested_scores_auc = []
nested_scores_acc = []
inner_cv_score = []
best_params_list = []

i=0
for train_index, test_index in outer_cv.split(X_train, y_train):
    X_outer_train, X_outer_test = X_train.iloc[train_index], X_train.iloc[test_index]
    y_outer_train, y_outer_test = y_train[train_index], y_train[test_index]

    # Inner CV for hyperparameter tuning
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    inner_search  = RandomizedSearchCV(
        RandomForestClassifier(random_state=123),
        param_distribution,
        n_iter=1, 
        cv=5,
        scoring='accuracy',
        random_state=123,
        n_jobs=-1
    )

    inner_search.fit(X_outer_train, y_outer_train)

    inner_cv_score.append(float(inner_search.best_score_))
    best_params_list.append(inner_search.best_params_)


    # Evaluate best model on outer test fold
    best_inner_model = clone(inner_search.best_estimator_)
    best_inner_model.fit(X_outer_train, y_outer_train)

    y_outer_proba = best_inner_model.predict_proba(X_outer_test)[:, 1]
    auc = roc_auc_score(y_outer_test, y_outer_proba)
    nested_scores_auc.append(auc)

    acc = accuracy_score(y_outer_test, best_inner_model.predict(X_outer_test))
    nested_scores_acc.append(acc)

    print(f'Fold {i} complete')
    i=i+1

print("\nNested CV (unbiased estimate):")
print("Inner fold accuracies:", inner_cv_score)
print("Outer fold accuracies:", nested_scores_acc)
print(f"Mean Accuracy: {np.mean(nested_scores_acc)} ± {np.std(nested_scores_acc)}")

print("Outer fold ROC-AUC:", nested_scores_auc)
print(f"Mean ROC-AUC: {np.mean(nested_scores_auc)} ± {np.std(nested_scores_auc)}")

# Summarize best parameters across folds
best_params_df = pd.DataFrame(best_params_list)
print("\nBest parameters per outer fold:")
print(best_params_df)

# Choose the most frequent or best-performing parameter set
final_best_params = best_params_df.mode().iloc[0].to_dict()
print("\nFinal chosen parameters for full training:")
print(final_best_params)



Fold 0 complete
Fold 1 complete
Fold 2 complete
Fold 3 complete
Fold 4 complete

Nested CV (unbiased estimate):
Inner fold accuracies: [0.8976225541763097, 0.9057647801388595, 0.8894172101830424, 0.9077635177782453, 0.9118241110877341]
Outer fold accuracies: [0.9344262295081968, 0.9098360655737705, 0.9180327868852459, 0.9016393442622951, 0.8852459016393442]
Mean Accuracy: 0.9098360655737705 ± 0.016393442622950838
Outer fold ROC-AUC: [0.9770114942528736, 0.9579638752052546, 0.9738372093023256, 0.9531653746770026, 0.9654392764857882]
Mean ROC-AUC: 0.9654834459846489 ± 0.009066002810869734

Best parameters per outer fold:
   bootstrap  max_depth  max_features  min_samples_leaf  min_samples_split  \
0       True         47      0.357525                 6                  4   
1       True         47      0.357525                 6                  4   
2       True         47      0.357525                 6                  4   
3       True         47      0.357525                 6      